# Exploratory Data Analysis — Adult Income Dataset

This notebook provides an interactive walkthrough of the data, model training,
SHAP baseline explanations, and robustness experiment results.

**Run `uv run python run_experiment.py` first** to generate all outputs.

## 1. Setup & Data Loading

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.data.load_data import load_raw_data

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (10, 6)

df = load_raw_data()
print(f"Dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

## 2. Data Overview

In [ ]:
df.info()

In [ ]:
# Missing values
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(f"Columns with missing values ({len(missing)}):\n")
for col, count in missing.items():
    print(f"  {col:<20s} {count:>6,}  ({count / len(df):.1%})")

In [ ]:
# Target distribution
target_counts = df["income"].value_counts()
print("Target distribution:\n")
for label, count in target_counts.items():
    print(f"  {label:<10s} {count:>6,}  ({count / len(df):.1%})")

target_counts.plot(kind="bar", color=["#4C72B0", "#C44E52"], alpha=0.8)
plt.title("Target Distribution (Income)")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Numeric feature distributions
from src.config import NUMERIC_FEATURES

df[NUMERIC_FEATURES].describe().T

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.flat, NUMERIC_FEATURES, strict=True):
    df[col].hist(bins=40, ax=ax, color="#4C72B0", alpha=0.7)
    ax.set_title(col)
plt.suptitle("Numeric Feature Distributions", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 3. Preprocessing

In [ ]:
from src.data.preprocess import run_preprocessing_pipeline

X_train, X_test, y_train, y_test, feature_names, preprocessor = run_preprocessing_pipeline(df)
print(f"\nFeatures: {len(feature_names)}")
print(f"Missing values after transform: {np.isnan(X_train).sum()}")

## 4. Model Training & Evaluation

In [ ]:
from src.models.train_models import train_all_models

models = train_all_models(X_train, y_train.values)

In [ ]:
from src.models.evaluate_models import evaluate_all_models

df_eval = evaluate_all_models(models, X_test, y_test.values)
df_eval

In [ ]:
# Visual comparison
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(df_eval))
width = 0.35
ax.bar(x - width / 2, df_eval["accuracy"], width, label="Accuracy", color="#4C72B0")
ax.bar(x + width / 2, df_eval["roc_auc"], width, label="ROC-AUC", color="#55A868")
ax.set_xticks(x)
ax.set_xticklabels(df_eval["model"])
ax.set_ylim(0.8, 1.0)
ax.set_ylabel("Score")
ax.set_title("Model Performance Comparison")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Baseline SHAP Explanations

In [ ]:
from src.config import N_LOCAL_SAMPLES
from src.explainability.compute_shap import compute_baseline_shap

baseline_shap = compute_baseline_shap(
    models, X_train, X_test, feature_names, n_local_samples=N_LOCAL_SAMPLES
)

In [ ]:
# Top-10 features per model
for name, res in baseline_shap.items():
    top10 = res["global_importance"]["feature_names_sorted"][:10]
    print(f"\n{name} — Top 10 most important features:")
    for i, feat in enumerate(top10, 1):
        print(f"  {i:>2}. {feat}")

In [ ]:
import shap

# SHAP summary plot for XGBoost (inline)
shap.summary_plot(
    baseline_shap["XGBoost"]["shap_values_test"],
    X_test,
    feature_names=feature_names,
    max_display=15,
    show=True,
)

## 6. Robustness Results

Load pre-computed metrics (from `run_experiment.py`).

In [ ]:
from src.config import METRICS_DIR

try:
    df_spearman = pd.read_csv(METRICS_DIR / "robustness_spearman.csv")
    df_cosine = pd.read_csv(METRICS_DIR / "robustness_cosine.csv")
    df_si = pd.read_csv(METRICS_DIR / "robustness_stability_index.csv")
    df_var = pd.read_csv(METRICS_DIR / "robustness_shap_variance.csv")
    print("Loaded robustness metrics.")
except FileNotFoundError:
    print("Robustness metrics not found. Run `uv run python run_experiment.py` first.")

In [ ]:
# Stability Index summary
print("Stability Index (S = 0.5 × Spearman + 0.5 × Cosine):\n")
pivot = df_si.pivot(index="model", columns="perturbation", values="stability_index")
pivot.style.format("{:.4f}").background_gradient(cmap="RdYlGn", vmin=0.5, vmax=1.0)

In [ ]:
# Spearman correlation distribution
mask = df_spearman["perturbation"].isin(["seed", "bootstrap", "noise"])
summary = (
    df_spearman[mask]
    .groupby(["model", "perturbation"])["spearman_rho"]
    .agg(["mean", "std", "min", "max"])
    .round(4)
)
summary

In [ ]:
# Cosine similarity distribution
mask = df_cosine["perturbation"].isin(["seed", "bootstrap", "noise"])
summary_cos = (
    df_cosine[mask]
    .groupby(["model", "perturbation"])["cosine_similarity"]
    .agg(["mean", "std", "min", "max"])
    .round(4)
)
summary_cos

In [ ]:
# Top-10 most unstable features (highest SHAP variance) per model
for model in df_var["model"].unique():
    model_df = df_var[df_var["model"] == model]
    avg_var = model_df.groupby("feature")["shap_variance"].mean().nlargest(10)
    print(f"\n{model} — Top 10 most unstable features (avg SHAP variance):")
    for i, (feat, var) in enumerate(avg_var.items(), 1):
        print(f"  {i:>2}. {feat:<45s} {var:.6f}")

## 7. Key Takeaways

After running the full experiment, review:

- **Which model produces the most stable SHAP explanations?** — Compare the Stability Index across models.
- **Which perturbation type causes the most instability?** — Look at Spearman ρ and cosine similarity per perturbation.
- **Which features are most sensitive to perturbations?** — High SHAP variance indicates explanation fragility.
- **Does model complexity correlate with explanation stability?** — Compare LR (linear) vs. RF/XGBoost (tree-based).